In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path("..") / "data"
CLEAN = (DATA_DIR / "clean_sales.parquet").as_posix()

print(Path.cwd())

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE TABLE clean_sales AS SELECT * FROM read_parquet('{CLEAN}')")
print(con.execute("SELECT COUNT(*) FROM clean_sales").fetchall()[0][0], "rows loaded")

C:\Users\kanak\retail-customer-analytics\notebooks
1021272 rows loaded


In [2]:
from pathlib import Path

print(Path.cwd())
for p in sorted((Path("..") / "data").glob("*")):
    print(p.name)

C:\Users\kanak\retail-customer-analytics\notebooks
clean_sales.parquet
cohort_retention.csv
excluded_rows.parquet
monthly_summary.csv
online_retail_II.xlsx
online_retail_II_raw.parquet
rfm_segments.csv


In [3]:
SNAPSHOT = con.execute(
    "SELECT CAST(CAST(MAX(invoice_ts) AS DATE) + 1 AS VARCHAR) FROM clean_sales"
).fetchall()[0][0]
print(SNAPSHOT)

2011-12-10


In [4]:
con.execute(f"""
CREATE OR REPLACE TABLE rfm_base AS
SELECT
    customer_id,
    DATE_DIFF('day',
              CAST(MAX(invoice_ts) FILTER (WHERE NOT is_return) AS DATE),
              DATE '{SNAPSHOT}')                                 AS recency_days,
    COUNT(DISTINCT invoice) FILTER (WHERE NOT is_return)         AS frequency,
    ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 2)       AS monetary_gross,
    ROUND(SUM(line_value), 2)                                    AS monetary_net
FROM clean_sales
WHERE customer_id IS NOT NULL
GROUP BY customer_id
HAVING COUNT(*) FILTER (WHERE NOT is_return) > 0
""")
print(con.execute("SELECT COUNT(*) FROM rfm_base").fetchall()[0][0], "customers")

5852 customers


In [5]:
df_rfm = con.execute("SELECT * FROM rfm_base").fetchdf()
print(df_rfm.describe().round(2))

print("Customers with net spend <= 0:", (df_rfm["monetary_net"] <= 0).sum())

con.sql("SELECT * FROM rfm_base ORDER BY monetary_net DESC LIMIT 10").show()
con.sql("SELECT * FROM rfm_base WHERE customer_id IN (12346, 16446)").show()

       customer_id  recency_days  frequency  monetary_gross  monetary_net
count      5852.00       5852.00    5852.00         5852.00       5852.00
mean      15319.71        200.73       6.25         2916.71       2795.63
std        1714.93        208.52      12.75        14306.87      13825.26
min       12346.00          1.00       1.00            2.95      -1343.24
25%       13837.75         26.00       1.00          339.58        330.94
50%       15320.50         96.00       3.00          856.02        840.24
75%       16802.25        380.00       7.00         2241.03       2172.03
max       18287.00        739.00     373.00       580987.04     578408.64
Customers with net spend <= 0: 20
┌─────────────┬──────────────┬───────────┬────────────────┬──────────────┐
│ customer_id │ recency_days │ frequency │ monetary_gross │ monetary_net │
│    int64    │    int64     │   int64   │     double     │    double    │
├─────────────┼──────────────┼───────────┼────────────────┼──────────────┤


In [6]:
con.sql("""
SELECT
    r.customer_id,
    r.invoice AS return_invoice,
    s.invoice AS sale_invoice,
    COUNT(*) AS matched_lines,
    ROUND(SUM(s.line_value), 2) AS sale_value
FROM clean_sales r
JOIN clean_sales s
  ON r.customer_id = s.customer_id
 AND r.stock_code  = s.stock_code
 AND r.quantity    = -s.quantity
 AND r.unit_price  = s.unit_price
 AND s.invoice_ts <= r.invoice_ts
 AND r.invoice_ts - s.invoice_ts <= INTERVAL 1 DAY
WHERE r.is_return AND NOT s.is_return AND r.customer_id IS NOT NULL
GROUP BY 1, 2, 3
ORDER BY sale_value DESC
LIMIT 15
""").show()

┌─────────────┬────────────────┬──────────────┬───────────────┬────────────┐
│ customer_id │ return_invoice │ sale_invoice │ matched_lines │ sale_value │
│    int64    │    varchar     │   varchar    │     int64     │   double   │
├─────────────┼────────────────┼──────────────┼───────────────┼────────────┤
│       16446 │ C581484        │ 581483       │             1 │   168469.6 │
│       12346 │ C541433        │ 541431       │             1 │    77183.6 │
│       16754 │ C513771        │ 513653       │            14 │    7500.32 │
│       17450 │ C529352        │ 529350       │             9 │    6325.08 │
│       12931 │ C508132        │ 508096       │            11 │     4815.8 │
│       12931 │ C508134        │ 508095       │            11 │     4798.8 │
│       14911 │ C562375        │ 562374       │            31 │     4345.1 │
│       12931 │ C508134        │ 508096       │            10 │     4203.8 │
│       12931 │ C508132        │ 508095       │            10 │     4203.8 │

In [7]:
VOID_INVOICES = ["541431", "C541433", "581483", "C581484"]
voids = ",".join(f"'{i}'" for i in VOID_INVOICES)

con.execute(f"""
CREATE OR REPLACE TABLE rfm_base AS
SELECT
    customer_id,
    DATE_DIFF('day',
              CAST(MAX(invoice_ts) FILTER (WHERE NOT is_return) AS DATE),
              DATE '{SNAPSHOT}')                                 AS recency_days,
    COUNT(DISTINCT invoice) FILTER (WHERE NOT is_return)         AS frequency,
    ROUND(SUM(line_value) FILTER (WHERE NOT is_return), 2)       AS monetary_gross,
    ROUND(SUM(line_value), 2)                                    AS monetary_net
FROM clean_sales
WHERE customer_id IS NOT NULL
  AND invoice NOT IN ({voids})
GROUP BY customer_id
HAVING COUNT(*) FILTER (WHERE NOT is_return) > 0
""")
print(con.execute("SELECT COUNT(*) FROM rfm_base").fetchall()[0][0], "customers")

5852 customers


In [8]:
con.sql("SELECT * FROM rfm_base WHERE customer_id IN (12346, 16446)").show()

┌─────────────┬──────────────┬───────────┬────────────────┬──────────────┐
│ customer_id │ recency_days │ frequency │ monetary_gross │ monetary_net │
│    int64    │    int64     │   int64   │     double     │    double    │
├─────────────┼──────────────┼───────────┼────────────────┼──────────────┤
│       16446 │          206 │         1 │            2.9 │          2.9 │
│       12346 │          530 │         2 │         169.36 │       169.36 │
└─────────────┴──────────────┴───────────┴────────────────┴──────────────┘



In [9]:
con.sql("""
SELECT
    quantile_cont(recency_days, [0.2, 0.4, 0.6, 0.8]) AS recency_cuts,
    quantile_cont(frequency,    [0.2, 0.4, 0.6, 0.8]) AS frequency_cuts,
    quantile_cont(monetary_net, [0.2, 0.4, 0.6, 0.8]) AS monetary_cuts
FROM rfm_base
""").show()

┌────────────────────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────┐
│        recency_cuts        │    frequency_cuts    │                           monetary_cuts                            │
│          double[]          │       double[]       │                              double[]                              │
├────────────────────────────┼──────────────────────┼────────────────────────────────────────────────────────────────────┤
│ [20.0, 59.0, 187.0, 409.0] │ [1.0, 2.0, 4.0, 8.0] │ [271.77, 590.6920000000001, 1178.3139999999999, 2827.158000000002] │
└────────────────────────────┴──────────────────────┴────────────────────────────────────────────────────────────────────┘



In [10]:
con.sql("""
SELECT frequency,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_customers
FROM rfm_base
GROUP BY frequency
ORDER BY frequency
LIMIT 10
""").show()

┌───────────┬───────────┬──────────────────┐
│ frequency │ customers │ pct_of_customers │
│   int64   │   int64   │      double      │
├───────────┼───────────┼──────────────────┤
│         1 │      1619 │             27.7 │
│         2 │       945 │             16.1 │
│         3 │       658 │             11.2 │
│         4 │       489 │              8.4 │
│         5 │       355 │              6.1 │
│         6 │       278 │              4.8 │
│         7 │       218 │              3.7 │
│         8 │       174 │              3.0 │
│         9 │       155 │              2.6 │
│        10 │        97 │              1.7 │
└───────────┴───────────┴──────────────────┘
  10 rows                        3 columns



In [11]:
cuts = con.execute("""
SELECT quantile_cont(recency_days, [0.2, 0.4, 0.6, 0.8]),
       quantile_cont(frequency,    [0.2, 0.4, 0.6, 0.8]),
       quantile_cont(monetary_net, [0.2, 0.4, 0.6, 0.8])
FROM rfm_base
""").fetchall()[0]
r_cuts, f_cuts, m_cuts = cuts
print("recency cuts:  ", r_cuts)
print("frequency cuts:", f_cuts)
print("monetary cuts: ", m_cuts)

con.execute(f"""
CREATE OR REPLACE TABLE rfm_scores AS
SELECT *,
    CASE WHEN recency_days <= {r_cuts[0]} THEN 5
         WHEN recency_days <= {r_cuts[1]} THEN 4
         WHEN recency_days <= {r_cuts[2]} THEN 3
         WHEN recency_days <= {r_cuts[3]} THEN 2
         ELSE 1 END AS r_score,
    CASE WHEN frequency <= {f_cuts[0]} THEN 1
         WHEN frequency <= {f_cuts[1]} THEN 2
         WHEN frequency <= {f_cuts[2]} THEN 3
         WHEN frequency <= {f_cuts[3]} THEN 4
         ELSE 5 END AS f_score,
    CASE WHEN monetary_net <= {m_cuts[0]} THEN 1
         WHEN monetary_net <= {m_cuts[1]} THEN 2
         WHEN monetary_net <= {m_cuts[2]} THEN 3
         WHEN monetary_net <= {m_cuts[3]} THEN 4
         ELSE 5 END AS m_score
FROM rfm_base
""")
print(con.execute("SELECT COUNT(*) FROM rfm_scores").fetchall()[0][0], "customers scored")

recency cuts:   [20.0, 59.0, 187.0, 409.0]
frequency cuts: [1.0, 2.0, 4.0, 8.0]
monetary cuts:  [271.77, 590.6920000000001, 1178.3139999999999, 2827.158000000002]
5852 customers scored


In [12]:
for col in ["r_score", "f_score", "m_score"]:
    con.sql(f"""
    SELECT '{col}' AS dimension, {col} AS score, COUNT(*) AS customers,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM rfm_scores GROUP BY {col} ORDER BY {col}
    """).show()

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├───────────┼───────┼───────────┼────────┤
│ r_score   │     1 │      1169 │   20.0 │
│ r_score   │     2 │      1171 │   20.0 │
│ r_score   │     3 │      1163 │   19.9 │
│ r_score   │     4 │      1135 │   19.4 │
│ r_score   │     5 │      1214 │   20.7 │
└───────────┴───────┴───────────┴────────┘

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├───────────┼───────┼───────────┼────────┤
│ f_score   │     1 │      1619 │   27.7 │
│ f_score   │     2 │       945 │   16.1 │
│ f_score   │     3 │      1147 │   19.6 │
│ f_score   │     4 │      1025 │   17.5 │
│ f_score   │     5 │      1116 │   19.1 │
└───────────┴───────┴───────────┴────────┘

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├────────

In [13]:
for col in ["r_score", "f_score", "m_score"]:
    con.sql(f"""
    SELECT '{col}' AS dimension, {col} AS score, COUNT(*) AS customers,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM rfm_scores GROUP BY {col} ORDER BY {col}
    """).show()

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├───────────┼───────┼───────────┼────────┤
│ r_score   │     1 │      1169 │   20.0 │
│ r_score   │     2 │      1171 │   20.0 │
│ r_score   │     3 │      1163 │   19.9 │
│ r_score   │     4 │      1135 │   19.4 │
│ r_score   │     5 │      1214 │   20.7 │
└───────────┴───────┴───────────┴────────┘

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├───────────┼───────┼───────────┼────────┤
│ f_score   │     1 │      1619 │   27.7 │
│ f_score   │     2 │       945 │   16.1 │
│ f_score   │     3 │      1147 │   19.6 │
│ f_score   │     4 │      1025 │   17.5 │
│ f_score   │     5 │      1116 │   19.1 │
└───────────┴───────┴───────────┴────────┘

┌───────────┬───────┬───────────┬────────┐
│ dimension │ score │ customers │  pct   │
│  varchar  │ int32 │   int64   │ double │
├────────

In [14]:
con.sql("""
SELECT m_score, COUNT(*) AS customers, MIN(monetary_net) AS lowest_net
FROM rfm_scores
WHERE monetary_net <= 0
GROUP BY m_score
""").show()

┌─────────┬───────────┬────────────┐
│ m_score │ customers │ lowest_net │
│  int32  │   int64   │   double   │
├─────────┼───────────┼────────────┤
│       1 │        20 │   -1343.24 │
└─────────┴───────────┴────────────┘



In [15]:
con.execute("""
CREATE OR REPLACE TABLE rfm_segments AS
WITH s AS (
    SELECT *,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
            WHEN f_score >= 4 AND r_score >= 3                  THEN 'Loyal'
            WHEN f_score >= 4 AND r_score <= 2                  THEN 'At risk (was frequent)'
            WHEN r_score >= 4 AND f_score = 1                   THEN 'New customers'
            WHEN r_score >= 4 AND f_score IN (2, 3)             THEN 'Potential loyalists'
            WHEN r_score = 3                                    THEN 'Needs attention'
            WHEN r_score = 2                                    THEN 'Hibernating'
            ELSE 'Lost'
        END AS segment
    FROM rfm_scores
)
SELECT *,
    CASE segment
        WHEN 'Champions' THEN 1 WHEN 'Loyal' THEN 2
        WHEN 'At risk (was frequent)' THEN 3 WHEN 'New customers' THEN 4
        WHEN 'Potential loyalists' THEN 5 WHEN 'Needs attention' THEN 6
        WHEN 'Hibernating' THEN 7 ELSE 8
    END AS segment_order
FROM s
""")
print(con.execute("SELECT COUNT(*) FROM rfm_segments").fetchall()[0][0], "customers segmented")

5852 customers segmented


In [16]:
con.sql("""
SELECT segment,
       COUNT(*)                                                       AS customers,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)             AS pct_customers,
       ROUND(SUM(monetary_net), 0)                                    AS net_spend,
       ROUND(100.0 * SUM(monetary_net) / SUM(SUM(monetary_net)) OVER (), 1) AS pct_net_spend,
       ROUND(AVG(recency_days), 0)                                    AS avg_recency_days,
       ROUND(AVG(frequency), 1)                                       AS avg_orders,
       ROUND(MEDIAN(monetary_net), 0)                                 AS median_net_spend
FROM rfm_segments
GROUP BY segment, segment_order
ORDER BY segment_order
""").show()

print("Total net spend, rfm_base:", con.execute("SELECT ROUND(SUM(monetary_net), 2) FROM rfm_base").fetchall()[0][0])

┌────────────────────────┬───────────┬───────────────┬────────────┬───────────────┬──────────────────┬────────────┬──────────────────┐
│        segment         │ customers │ pct_customers │ net_spend  │ pct_net_spend │ avg_recency_days │ avg_orders │ median_net_spend │
│        varchar         │   int64   │    double     │   double   │    double     │      double      │   double   │      double      │
├────────────────────────┼───────────┼───────────────┼────────────┼───────────────┼──────────────────┼────────────┼──────────────────┤
│ Champions              │      1257 │          21.5 │ 11286941.0 │          69.0 │             20.0 │       17.3 │           3911.0 │
│ Loyal                  │       598 │          10.2 │  1681684.0 │          10.3 │             86.0 │        8.3 │           1863.0 │
│ At risk (was frequent) │       286 │           4.9 │   911043.0 │           5.6 │            339.0 │        8.6 │           1720.0 │
│ New customers          │       239 │           4.1 │ 

In [17]:
con.sql("""
SELECT COUNT(*) AS customers, ROUND(SUM(net), 2) AS net_spend
FROM (
    SELECT customer_id, SUM(line_value) AS net
    FROM clean_sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
    HAVING COUNT(*) FILTER (WHERE NOT is_return) = 0
)
""").show()

┌───────────┬───────────┐
│ customers │ net_spend │
│   int64   │  double   │
├───────────┼───────────┤
│        23 │  -1406.13 │
└───────────┴───────────┘



In [18]:
con.sql("""
WITH ranked AS (
    SELECT monetary_net,
           row_number() OVER (ORDER BY monetary_net DESC) AS rnk,
           COUNT(*) OVER () AS n
    FROM rfm_base
)
SELECT
    ROUND(100.0 * SUM(monetary_net) FILTER (WHERE rnk <= n * 0.2) / SUM(monetary_net), 1) AS top_20pct_share,
    ROUND(100.0 * SUM(monetary_net) FILTER (WHERE rnk <= n * 0.1) / SUM(monetary_net), 1) AS top_10pct_share,
    ROUND(100.0 * SUM(monetary_net) FILTER (WHERE rnk <= 2)       / SUM(monetary_net), 1) AS top_2_customers_share
FROM ranked
""").show()

┌─────────────────┬─────────────────┬───────────────────────┐
│ top_20pct_share │ top_10pct_share │ top_2_customers_share │
│     double      │     double      │        double         │
├─────────────────┼─────────────────┼───────────────────────┤
│            76.8 │            63.2 │                   6.7 │
└─────────────────┴─────────────────┴───────────────────────┘



In [19]:
con.execute("UPDATE rfm_segments SET segment = 'Recent one-time buyers' WHERE segment = 'New customers'")
print(con.execute("SELECT COUNT(*) FROM rfm_segments WHERE segment = 'Recent one-time buyers'").fetchall()[0][0])

239


In [20]:
from pathlib import Path

OUT = (Path("..") / "data" / "rfm_segments.csv").as_posix()
con.execute(f"COPY rfm_segments TO '{OUT}' (FORMAT CSV, HEADER)")
print("saved", OUT)

saved ../data/rfm_segments.csv
